# YouTube Recommendation Data Cleaning and Preparation

## Project Overview

This project uses a YouTube recommendation dataset containing user viewing and engagement activity. The objective is to clean and prepare the dataset in Python before importing it into Power BI.

The final Power BI dashboard will analyze:

* Recommendation and click performance
* User engagement through likes, comments, and subscriptions
* Watch time and video completion
* Performance by video category
* Performance by device and time of day
* Trends in user activity over time

## Data-Cleaning Objectives

The following steps will be performed:

1. Load and inspect the dataset.
2. Identify missing values and duplicate records.
3. Correct inconsistent values in the `liked` column.
4. Standardize video category names.
5. Convert `timestamp` into a valid datetime field.
6. handle invalid and missing timestamps.
7. Identify negative or impossible watch-time values.
8. Ensure watch time does not exceed video duration.
9. Recalculate `watch_percent`.
10. Validate binary columns such as `clicked`, `recommended`, `commented`, and `subscribed_after`.
11. Create useful date fields for Power BI analysis.
12. Export the cleaned dataset for dashboard development.

## Expected Output

The final output will be a clean and validated dataset suitable for building an AI-powered YouTube Recommendation Performance Dashboard in Power BI. An agentic AI component will later be added to detect unusual KPI changes, investigate possible causes, summarize insights, and recommend business actions.



In [1]:
# Import Libraries
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [5]:
# Load the dataset
youtube_df = pd.read_csv(r"C:\Users\prati\Downloads\youtube recommendation dataset.csv")

# Display the first five rows
youtube_df.head()

,user_id,video_id,video_duration,watch_time,liked,commented,subscribed_after,category,device,watch_time_of_day,recommended,clicked,timestamp,watch_percent
0,88263,19387,1499,"1,499.00",1,0,0,News,TV,Night,0,0,2025-07-16 06:10:54,1.00
1,46796,5150,2955,"2,955.00",1,0,0,Education,Mobile,Night,1,1,2024-07-09 10:22:22,1.00
2,77686,3172,389,389.00,0,0,0,Sports,Desktop,Afternoon,0,0,2025-02-07 09:46:36,1.00
3,26723,27669,1231,"1,231.00",1,1,0,Gaming,TV,Evening,0,0,2024-04-28 08:07:13,1.00
4,55299,24612,3573,353.00,0,1,0,Lifestyle,Desktop,Evening,1,0,2024-03-30 12:54:24,0.10


In [6]:
# Check dataset size and columns
print(f"Number of rows: {youtube_df.shape[0]:,}")
print(f"Number of columns: {youtube_df.shape[1]}")

print("\nColumn names:")
print(youtube_df.columns.tolist())

Number of rows: 1,000,000
Number of columns: 14

Column names:
['user_id', 'video_id', 'video_duration', 'watch_time', 'liked', 'commented', 'subscribed_after', 'category', 'device', 'watch_time_of_day', 'recommended', 'clicked', 'timestamp', 'watch_percent']


In [7]:
# Inspect data types and missing values
youtube_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 14 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   user_id            1000000 non-null  int64  
 1   video_id           1000000 non-null  int64  
 2   video_duration     1000000 non-null  int64  
 3   watch_time         1000000 non-null  float64
 4   liked              997952 non-null   object 
 5   commented          1000000 non-null  int64  
 6   subscribed_after   1000000 non-null  int64  
 7   category           1000000 non-null  object 
 8   device             1000000 non-null  object 
 9   watch_time_of_day  1000000 non-null  object 
 10  recommended        1000000 non-null  int64  
 11  clicked            1000000 non-null  int64  
 12  timestamp          1000000 non-null  object 
 13  watch_percent      1000000 non-null  float64
dtypes: float64(2), int64(7), object(5)
memory usage: 106.8+ MB


In [9]:
# Analyze missing values
# Check missing values
youtube_df.isnull().sum()

user_id                 0
video_id                0
video_duration          0
watch_time              0
liked                2048
commented               0
subscribed_after        0
category                0
device                  0
watch_time_of_day       0
recommended             0
clicked                 0
timestamp               0
watch_percent           0
dtype: int64

In [10]:
# Check values in the liked column
youtube_df["liked"].value_counts(dropna=False)

liked
0      696155
1      298845
NaN      2048
2        1007
no        988
yes       957
Name: count, dtype: int64

In [14]:
# Clean the 'liked' column
# Create a copy of the dataset
youtube_clean = youtube_df.copy()

# Standardize the liked values
youtube_clean["liked"] = youtube_clean["liked"].replace({
    "1": "Liked",
    1: "Liked",
    "yes": "Liked",
    "0": "Not Liked",
    0: "Not Liked",
    "no": "Not Liked",
    "2": "Unknown",
    2: "Unknown"
})

# Replace missing values
youtube_clean["liked"] = youtube_clean["liked"].fillna("Unknown")

In [12]:
youtube_clean["liked"].value_counts()

liked
Not Liked    697143
Liked        299802
Unknown        3055
Name: count, dtype: int64

In [13]:
# Inspect the video categories 
# Check the category values
youtube_clean["category"].value_counts()

category
Comedy       124467
News         123920
Music        123851
Tech         123735
Sports       123644
Gaming       123595
Lifestyle    123457
Education    123331
Tech           1749
gamingg        1734
MUsic          1663
Ed             1637
music          1629
COMEDY         1588
Name: count, dtype: int64

In [15]:
# Clean video categories
# Clean category values
youtube_clean["category"] = youtube_clean["category"].replace({
    "Tech ": "Tech",
    "gamingg": "Gaming",
    "MUsic": "Music",
    "music": "Music",
    "Ed": "Education",
    "COMEDY": "Comedy"
})

In [16]:
youtube_clean["category"].value_counts()

category
Music        127143
Comedy       126055
Tech         125484
Gaming       125329
Education    124968
News         123920
Sports       123644
Lifestyle    123457
Name: count, dtype: int64

In [17]:
# Convert timestamp to datetime
youtube_clean["timestamp"] = pd.to_datetime(
    youtube_clean["timestamp"],
    errors="coerce"
)

In [18]:
# Check the timestamp data type
youtube_clean["timestamp"].dtype

dtype('<M8[ns]')

In [19]:
# Count invalid timestamps
youtube_clean["timestamp"].isnull().sum()

np.int64(2000)

In [20]:
# Count negative watch-time values
(youtube_clean["watch_time"] < 0).sum()

np.int64(500)

In [21]:
# Count watch times greater than video duration
(youtube_clean["watch_time"] > youtube_clean["video_duration"]).sum()

np.int64(4500)

In [22]:
# Count invalid video durations
(youtube_clean["video_duration"] <= 0).sum()

np.int64(1000)

In [23]:
# Change invalid video durations to missing
youtube_clean.loc[
    youtube_clean["video_duration"] <= 0,
    "video_duration"
] = np.nan

# Change negative watch times to missing
youtube_clean.loc[
    youtube_clean["watch_time"] < 0,
    "watch_time"
] = np.nan

# Cap watch time at the video duration
youtube_clean.loc[
    youtube_clean["watch_time"] > youtube_clean["video_duration"],
    "watch_time"
] = youtube_clean["video_duration"]

In [24]:
# Recalculate watch percentage
youtube_clean["watch_percent"] = (
    youtube_clean["watch_time"] /
    youtube_clean["video_duration"]
)

In [25]:
# Verify the results
print("Invalid durations:",
      (youtube_clean["video_duration"] <= 0).sum())

print("Negative watch times:",
      (youtube_clean["watch_time"] < 0).sum())

print("Watch percentage above 1:",
      (youtube_clean["watch_percent"] > 1).sum())

Invalid durations: 0
Negative watch times: 0
Watch percentage above 1: 0


In [26]:
# Validate binary columns
# Check binary column values
youtube_clean["commented"].value_counts()

commented
0    900503
1     99497
Name: count, dtype: int64

In [27]:
# Check subscribed values
youtube_clean["subscribed_after"].value_counts()

subscribed_after
0    949996
1     50004
Name: count, dtype: int64

In [28]:
# Check recommended values
youtube_clean["recommended"].value_counts()

recommended
0    600232
1    399768
Name: count, dtype: int64

In [29]:
# Check clicked values
youtube_clean["clicked"].value_counts()

clicked
0    699229
1    300771
Name: count, dtype: int64

In [30]:
# Check device values
youtube_clean["device"].value_counts()

device
TV         250083
Tablet     250066
Desktop    249983
Mobile     249868
Name: count, dtype: int64

In [31]:
# Check time-of-day values
youtube_clean["watch_time_of_day"].value_counts()

watch_time_of_day
Morning      250258
Afternoon    250246
Evening      249822
Night        249674
Name: count, dtype: int64

In [32]:
# Count duplicate rows
youtube_clean.duplicated().sum()

np.int64(0)

In [33]:
# Create date columns for PowerbI
youtube_clean["date"] = youtube_clean["timestamp"].dt.date
youtube_clean["year"] = youtube_clean["timestamp"].dt.year
youtube_clean["month"] = youtube_clean["timestamp"].dt.month
youtube_clean["month_name"] = youtube_clean["timestamp"].dt.month_name()
youtube_clean["day_name"] = youtube_clean["timestamp"].dt.day_name()

In [34]:
# Check the new columns
youtube_clean[
    ["timestamp", "date", "year", "month", "month_name", "day_name"]
].head()

,timestamp,date,year,month,month_name,day_name
0,2025-07-16 06:10:54,2025-07-16,"2,025.00",7.00,July,Wednesday
1,2024-07-09 10:22:22,2024-07-09,"2,024.00",7.00,July,Tuesday
2,2025-02-07 09:46:36,2025-02-07,"2,025.00",2.00,February,Friday
3,2024-04-28 08:07:13,2024-04-28,"2,024.00",4.00,April,Sunday
4,2024-03-30 12:54:24,2024-03-30,"2,024.00",3.00,March,Saturday


In [35]:
# Remove decimals from year and month
youtube_clean["year"] = youtube_clean["year"].astype("Int64")
youtube_clean["month"] = youtube_clean["month"].astype("Int64")

In [37]:
youtube_clean[
    ["timestamp", "date", "year", "month", "month_name", "day_name"]
].head()

,timestamp,date,year,month,month_name,day_name
0,2025-07-16 06:10:54,2025-07-16,2025,7,July,Wednesday
1,2024-07-09 10:22:22,2024-07-09,2024,7,July,Tuesday
2,2025-02-07 09:46:36,2025-02-07,2025,2,February,Friday
3,2024-04-28 08:07:13,2024-04-28,2024,4,April,Sunday
4,2024-03-30 12:54:24,2024-03-30,2024,3,March,Saturday


In [38]:
# Check the final dataset size
youtube_clean.shape

(1000000, 19)

In [39]:
# Check remaining missing values
youtube_clean.isnull().sum()

user_id                 0
video_id                0
video_duration       1000
watch_time            500
liked                   0
commented               0
subscribed_after        0
category                0
device                  0
watch_time_of_day       0
recommended             0
clicked                 0
timestamp            2000
watch_percent        1500
date                 2000
year                 2000
month                2000
month_name           2000
day_name             2000
dtype: int64

In [40]:
# Check duplicate rows
youtube_clean.duplicated().sum()

np.int64(0)

In [42]:
# Export cleaned dataset
youtube_clean.to_csv(
    r"C:\Users\prati\Downloads\youtube_recommendation_cleaned.csv",
    index=False
)

print("Cleaned dataset exported successfully.")

Cleaned dataset exported successfully.


## Performance Insights

In [58]:
# Create a separate dataframe for analysis
analysis_df = youtube_clean.copy()

# Create quarter
analysis_df["quarter"] = analysis_df["timestamp"].dt.to_period("Q").astype(str)

# Create video-length groups
length_bins = [0, 600, 1200, 1800, 2400, 3000, 3600, np.inf]

length_labels = [
    "Under 10 min",
    "10–20 min",
    "20–30 min",
    "30–40 min",
    "40–50 min",
    "50–60 min",
    "Over 60 min"
]

analysis_df["video_length_group"] = pd.cut(
    ai_df["video_duration"],
    bins=length_bins,
    labels=length_labels,
    right=False
)

print("Analysis dataset prepared:", analysis_df.shape)

Analysis dataset prepared: (1000000, 21)


In [59]:
def calculate_ctr(data):
    recommended_impressions = data["recommended"].sum()

    recommended_clicks = data.loc[
        data["recommended"] == 1,
        "clicked"
    ].sum()

    if recommended_impressions == 0:
        return np.nan

    return recommended_clicks / recommended_impressions


overall_ctr = calculate_ctr(analysis_df)
overall_watch_completion = analysis_df["watch_percent"].mean()

print(f"Overall Recommendation CTR: {overall_ctr:.2%}")
print(f"Average Watch Completion: {overall_watch_completion:.2%}")

Overall Recommendation CTR: 29.99%
Average Watch Completion: 74.97%


In [60]:
# CTR by category

category_analysis = (
    analysis_df.groupby("category")
    .apply(calculate_ctr)
    .reset_index(name="recommendation_ctr")
)

category_analysis["variance_pp"] = (
    category_analysis["recommendation_ctr"] - overall_ctr
) * 100


# CTR by device and time of day

device_time_analysis = (
    analysis_df.groupby(["device", "watch_time_of_day"])
    .apply(calculate_ctr)
    .reset_index(name="recommendation_ctr")
)

device_time_analysis["variance_pp"] = (
    device_time_analysis["recommendation_ctr"] - overall_ctr
) * 100


# CTR by quarter

quarter_analysis = (
    analysis_df.dropna(subset=["timestamp"])
    .groupby("quarter")
    .apply(calculate_ctr)
    .reset_index(name="recommendation_ctr")
    .sort_values("quarter")
)

# Percentage version for easier validation
quarter_analysis["recommendation_ctr_percent"] = (
    quarter_analysis["recommendation_ctr"] * 100
).round(2)

# video-length classification

duration = analysis_df["video_duration"]

analysis_df["video_length_group"] = np.select(
    [
        duration.isna(),
        duration < 600,
        duration < 1200,
        duration < 1800,
        duration < 2400,
        duration < 3000,
        duration <= 3600
    ],
    [
        "Unknown",
        "Under 10 min",
        "10–20 min",
        "20–30 min",
        "30–40 min",
        "40–50 min",
        "50–60 min"
    ],
    default="Over 60 min"
)

# Watch completion by video length

video_length_order = [
    "Under 10 min",
    "10–20 min",
    "20–30 min",
    "30–40 min",
    "40–50 min",
    "50–60 min"
]

length_analysis = (
    analysis_df[
        analysis_df["video_length_group"].isin(video_length_order)
    ]
    .groupby("video_length_group")["watch_percent"]
    .mean()
    .reindex(video_length_order)
    .reset_index(name="watch_completion")
)

# Percentage version for easier validation
length_analysis["watch_completion_percent"] = (
    length_analysis["watch_completion"] * 100
).round(1)


# Display all analysis tables

display(category_analysis)
display(device_time_analysis)
display(quarter_analysis)
display(length_analysis)


C:\Users\prati\AppData\Local\Temp\ipykernel_12628\1237821722.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_ctr)
C:\Users\prati\AppData\Local\Temp\ipykernel_12628\1237821722.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_ctr)
C:\Users\prati\AppData\Local\Temp\ipykernel_12628\1237821722.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping col

,category,recommendation_ctr,variance_pp
0,Comedy,0.30,0.10
1,Education,0.30,0.19
2,Gaming,0.30,0.04
3,Lifestyle,0.30,-0.17
4,Music,0.30,-0.29
5,News,0.30,0.22
6,Sports,0.30,0.02
7,Tech,0.30,-0.11


,device,watch_time_of_day,recommendation_ctr,variance_pp
0,Desktop,Afternoon,0.30,-0.04
1,Desktop,Evening,0.30,-0.22
2,Desktop,Morning,0.30,-0.41
3,Desktop,Night,0.30,0.45
4,Mobile,Afternoon,0.30,0.32
5,Mobile,Evening,0.30,0.19
6,Mobile,Morning,0.30,-0.31
7,Mobile,Night,0.30,-0.07
8,TV,Afternoon,0.30,-0.39
9,TV,Evening,0.30,0.23


,quarter,recommendation_ctr,recommendation_ctr_percent
0,2023Q1,0.30,30.09
1,2023Q2,0.30,30.12
2,2023Q3,0.30,29.73
3,2023Q4,0.30,29.76
4,2024Q1,0.30,29.79
5,2024Q2,0.30,30.09
6,2024Q3,0.30,30.17
7,2024Q4,0.30,29.74
8,2025Q1,0.30,29.86
9,2025Q2,0.31,30.51


,video_length_group,watch_completion,watch_completion_percent
0,Under 10 min,0.96,95.80
1,10–20 min,0.88,87.60
2,20–30 min,0.79,79.20
3,30–40 min,0.71,71.10
4,40–50 min,0.63,62.70
5,50–60 min,0.54,54.40


In [61]:
# Identify strongest and weakest device-time combinations
strongest = device_time_analysis.loc[
    device_time_analysis["variance_pp"].idxmax()
]

weakest = device_time_analysis.loc[
    device_time_analysis["variance_pp"].idxmin()
]

# Compare latest quarter with previous quarter
latest_quarter = quarter_analysis.iloc[-1]
previous_quarter = quarter_analysis.iloc[-2]

quarter_change_pp = (
    latest_quarter["recommendation_ctr"]
    - previous_quarter["recommendation_ctr"]
) * 100

# Interpret the size of the quarterly movement
if abs(quarter_change_pp) < 0.50:
    trend_assessment = "a small movement"
else:
    trend_assessment = "a notable movement"

# Create the insight output
analysis_insights = pd.DataFrame([
    {
        "Priority": 1,
        "Insight_Type": "Status",
        "Headline": "Overall recommendation performance",
        "Insight": (
            f"Recommendation CTR is {overall_ctr:.1%}, while average "
            f"watch completion is {overall_watch_completion:.1%}."
        )
    },
    {
        "Priority": 2,
        "Insight_Type": "Strength",
        "Headline": "Strongest device-time segment",
        "Insight": (
            f"{strongest['device']} during "
            f"{strongest['watch_time_of_day']} has the highest CTR "
            f"variance at {strongest['variance_pp']:+.2f} percentage "
            f"points versus the overall CTR."
        )
    },
    {
        "Priority": 3,
        "Insight_Type": "Risk",
        "Headline": "Lowest device-time segment",
        "Insight": (
            f"{weakest['device']} during "
            f"{weakest['watch_time_of_day']} has the lowest CTR "
            f"variance at {weakest['variance_pp']:+.2f} percentage "
            f"points versus the overall CTR."
        )
    },
    {
        "Priority": 4,
        "Insight_Type": "Trend",
        "Headline": "Latest quarterly movement",
        "Insight": (
            f"Recommendation CTR changed by {quarter_change_pp:+.2f} "
            f"percentage points in {latest_quarter['quarter']}. "
            f"This is {trend_assessment}."
        )
    },
    {
        "Priority": 5,
        "Insight_Type": "Action",
        "Headline": "Recommended test",
        "Insight": (
            f"Investigate recommendation timing for "
            f"{weakest['device']} users during "
            f"{weakest['watch_time_of_day']}. Test changes before "
            f"drawing causal conclusions."
        )
    }
])

analysis_insights

,Priority,Insight_Type,Headline,Insight
0,1,Status,Overall recommendation performance,"Recommendation CTR is 30.0%, while average wat..."
1,2,Strength,Strongest device-time segment,Desktop during Night has the highest CTR varia...
2,3,Risk,Lowest device-time segment,Desktop during Morning has the lowest CTR vari...
3,4,Trend,Latest quarterly movement,Recommendation CTR changed by -0.42 percentage...
4,5,Action,Recommended test,Investigate recommendation timing for Desktop ...


In [62]:
for _, row in ai_insights.iterrows():
    print(f"{row['Priority']}. {row['Headline']}")
    print(row["Insight"])
    print()

1. Overall recommendation performance
Recommendation CTR is 30.0%, while average watch completion is 75.0%.

2. Strongest device-time segment
Desktop during Night has the highest CTR variance at +0.45 percentage points versus the overall CTR.

3. Lowest device-time segment
Desktop during Morning has the lowest CTR variance at -0.41 percentage points versus the overall CTR.

4. Latest quarterly movement
Recommendation CTR changed by -0.42 percentage points in 2025Q3. This is a small movement.

5. Category performance range
News has the highest CTR variance at +0.22 percentage points, while Music has the lowest at -0.29 percentage points. These differences are small.

6. Completion declines for longer videos
Under 10 min videos have 95.8% average completion, compared with 54.4% for 50–60 min videos. This is an association and does not establish causation.

7. Recommended test
Investigate recommendation timing for Desktop users during Morning. Test changes before drawing causal conclusion

In [68]:

# Additional dashboard insights
# Strongest and weakest categories
best_category = category_analysis.loc[
    category_analysis["variance_pp"].idxmax()
]

worst_category = category_analysis.loc[
    category_analysis["variance_pp"].idxmin()
]

# Highest and lowest watch-completion groups
best_length = length_analysis.loc[
    length_analysis["watch_completion"].idxmax()
]

lowest_length = length_analysis.loc[
    length_analysis["watch_completion"].idxmin()
]

additional_insights = pd.DataFrame([
    {
        "Priority": 5,
        "Insight_Type": "Category",
        "Headline": "Category performance range",
        "Insight": (
            f"{best_category['category']} has the highest CTR variance "
            f"at {best_category['variance_pp']:+.2f} percentage points, "
            f"while {worst_category['category']} has the lowest at "
            f"{worst_category['variance_pp']:+.2f} percentage points. "
            f"These differences are small."
        )
    },
    {
        "Priority": 6,
        "Insight_Type": "Engagement",
        "Headline": "Completion declines for longer videos",
        "Insight": (
            f"{best_length['video_length_group']} videos have "
            f"{best_length['watch_completion']:.1%} average completion, "
            f"compared with {lowest_length['watch_completion']:.1%} for "
            f"{lowest_length['video_length_group']} videos. This is an "
            f"association and does not establish causation."
        )
    }
])

# Move the recommended action to the final position
analysis_insights.loc[
    analysis_insights["Insight_Type"] == "Action",
    "Priority"
] = 7

# Combine and sort all insights
analysis_insights = (
    pd.concat(
        [analysis_insights, additional_insights],
        ignore_index=True
    )
    .sort_values("Priority")
    .reset_index(drop=True)
)

analysis_insights

,Priority,Insight_Type,Headline,Insight
0,1,Status,Overall recommendation performance,"Recommendation CTR is 30.0%, while average wat..."
1,2,Strength,Strongest device-time segment,Desktop during Night has the highest CTR varia...
2,3,Risk,Lowest device-time segment,Desktop during Morning has the lowest CTR vari...
3,4,Trend,Latest quarterly movement,Recommendation CTR changed by -0.42 percentage...
4,5,Category,Category performance range,News has the highest CTR variance at +0.22 per...
5,5,Category,Category performance range,News has the highest CTR variance at +0.22 per...
6,5,Category,Category performance range,News has the highest CTR variance at +0.22 per...
7,6,Engagement,Completion declines for longer videos,Under 10 min videos have 95.8% average complet...
8,6,Engagement,Completion declines for longer videos,Under 10 min videos have 95.8% average complet...
9,6,Engagement,Completion declines for longer videos,Under 10 min videos have 95.8% average complet...


In [64]:
for _, row in analysis_insights.iterrows():
    print(f"{row['Priority']}. {row['Headline']}")
    print(row["Insight"])
    print()

1. Overall recommendation performance
Recommendation CTR is 30.0%, while average watch completion is 75.0%.

2. Strongest device-time segment
Desktop during Night has the highest CTR variance at +0.45 percentage points versus the overall CTR.

3. Lowest device-time segment
Desktop during Morning has the lowest CTR variance at -0.41 percentage points versus the overall CTR.

4. Latest quarterly movement
Recommendation CTR changed by -0.42 percentage points in 2025Q3. This is a small movement.

5. Category performance range
News has the highest CTR variance at +0.22 percentage points, while Music has the lowest at -0.29 percentage points. These differences are small.

5. Category performance range
News has the highest CTR variance at +0.22 percentage points, while Music has the lowest at -0.29 percentage points. These differences are small.

6. Completion declines for longer videos
Under 10 min videos have 95.8% average completion, compared with 54.4% for 50–60 min videos. This is an ass

In [65]:
# Add Automated Refresh Function

from datetime import datetime
from pathlib import Path


def refresh_analysis_insights(analysis_insights, output_folder):
    """
    Validates, timestamps and exports the latest insights.
    """

    required_columns = {
        "Priority",
        "Insight_Type",
        "Headline",
        "Insight"
    }

    missing_columns = required_columns - set(analysis_insights.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    # Remove blank and duplicate insights
    final_insights = (
        analysis_insights
        .dropna(subset=["Headline", "Insight"])
        .drop_duplicates(subset=["Headline", "Insight"])
        .sort_values("Priority")
        .reset_index(drop=True)
        .copy()
    )

    # Add refresh information
    refresh_time = datetime.now()

    final_insights["Generated_At"] = (
        refresh_time.strftime("%Y-%m-%d %H:%M:%S")
    )

    final_insights["Data_Type"] = "Synthetic portfolio data"
    final_insights["Agent_Version"] = "1.0"

    # Create the output folder if it does not exist
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    output_file = output_folder / "Analysis_Insights.csv"

    final_insights.to_csv(output_file, index=False)

    print("Analysis insight refresh completed successfully.")
    print(f"Insights generated: {len(final_insights)}")
    print(f"Output file: {output_file}")
    print(f"Generated at: {refresh_time:%Y-%m-%d %H:%M:%S}")

    return final_insights

In [66]:
# Export
from pathlib import Path
import os

# Locate OneDrive folder
onedrive_folder = (
    os.environ.get("OneDrive")
    or os.environ.get("OneDriveConsumer")
)

output_folder = (
    Path(onedrive_folder)
    / "Documents"
    / "SU"
    / "My_Projects"
    / "Power BI"
    / "Automation"
)

final_analysis_insights = refresh_analysis_insights(
    analysis_insights=analysis_insights,
    output_folder=output_folder
)

output_file = output_folder / "Analysis_Insights.csv"

print("Saved to:", output_file)
print("File exists:", output_file.exists())

# Open the correct folder
os.startfile(output_folder)

Analysis insight refresh completed successfully.
Insights generated: 7
Output file: C:\Users\prati\OneDrive\Documents\SU\My_Projects\Power BI\Automation\Analysis_Insights.csv
Generated at: 2026-09-14 21:22:06
Saved to: C:\Users\prati\OneDrive\Documents\SU\My_Projects\Power BI\Automation\Analysis_Insights.csv
File exists: True
